In [1]:
import math

def foil_gain(p0, n0, p1, n1):
    if p1 == 0 or p1 + n1 == 0 or p0 == 0 or p0 + n0 == 0:
        return -float("inf")
    return p1 * (
        math.log2(p1 / (p1 + n1)) -
        math.log2(p0 / (p0 + n0))
    )

def covers(example, rule):
    for attribute, value in rule.items():
        if example[attribute] != value:
            return False
    return True

def foil(examples, target):
    Pos = [x for x in examples if x["Target"] == target]
    Neg = [x for x in examples if x["Target"] != target]
    learned_rules = []

    while Pos:
        NewRule = {}
        NewRuleNeg = Neg.copy()

        while NewRuleNeg:
            best_literal = None
            best_gain = -float("inf")

            attributes = [
                key for key in examples[0]
                if key != "Target" and key not in NewRule
            ]

            for attribute in attributes:
                values = set(x[attribute] for x in examples)

                for value in values:
                    p1 = sum(1 for x in Pos if x[attribute] == value)
                    n1 = sum(1 for x in NewRuleNeg if x[attribute] == value)

                    p0 = len(Pos)
                    n0 = len(NewRuleNeg)

                    gain = foil_gain(p0, n0, p1, n1)

                    if gain > best_gain:
                        best_gain = gain
                        best_literal = (attribute, value)

            if best_literal is None:
                break

            attribute, value = best_literal
            NewRule[attribute] = value

            NewRuleNeg = [
                x for x in NewRuleNeg
                if x[attribute] == value
            ]

        learned_rules.append(NewRule)

        Pos = [
            x for x in Pos
            if not covers(x, NewRule)
        ]

    return learned_rules

examples = [
    {"Sky": "Sunny", "Temp": "Warm", "Humidity": "Normal", "Target": "Yes"},
    {"Sky": "Sunny", "Temp": "Warm", "Humidity": "High", "Target": "No"},
    {"Sky": "Cloudy", "Temp": "Warm", "Humidity": "High", "Target": "Yes"},
    {"Sky": "Rainy", "Temp": "Cool", "Humidity": "High", "Target": "No"},
    {"Sky": "Sunny", "Temp": "Cool", "Humidity": "Normal", "Target": "Yes"},
    {"Sky": "Rainy", "Temp": "Warm", "Humidity": "Normal", "Target": "Yes"},
    {"Sky": "Rainy", "Temp": "Cool", "Humidity": "Normal", "Target": "No"}
]

rules = foil(examples, "Yes")

for i, rule in enumerate(rules, 1):
    conditions = [f"{a} = {v}" for a, v in rule.items()]
    print(f"Rule {i}: IF " + " AND ".join(conditions) + " THEN Target = Yes")

Rule 1: IF Temp = Warm AND Humidity = Normal THEN Target = Yes
Rule 2: IF Sky = Cloudy THEN Target = Yes
Rule 3: IF Sky = Sunny AND Temp = Cool THEN Target = Yes
